# Ecuaciones diferenciales ordinarias · 2026-2

Universidad del Pacífico. Clase basada en el capítulo 8 de las notas del curso.

Problema de Cauchy → forma integral → Euler y consistencia → trapecio/Heun → punto medio → RK4.

Ejecute las celdas en orden. Solo se necesita **NumPy**. Las salidas de esta versión
ya fueron ejecutadas. `y(t_k)` denota la solución exacta y `y_k` la aproximación.
En las notas, **Euler mejorado** corresponde al **punto medio explícito**.
El trapecio implícito y Heun son fórmulas diferentes.

El ejemplo principal es $y'=y-t^2+1$, $y(0)=0.5$, con solución exacta
$y(t)=(t+1)^2-e^t/2$. Se compara el error máximo **en los nodos** de $[0,2]$.
Los parámetros del modelo de capital son hipotéticos.


## Métodos de un paso


In [1]:
from pathlib import Path
import json
import numpy as np


def paso_euler(f, t, y, h):
    return y + h * f(t, y)


def paso_punto_medio(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h / 2, y + h * k1 / 2)
    return y + h * k2


def paso_heun(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h, y + h * k1)
    return y + h * (k1 + k2) / 2


def paso_rk4(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h / 2, y + h * k1 / 2)
    k3 = f(t + h / 2, y + h * k2 / 2)
    k4 = f(t + h, y + h * k3)
    return y + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6


METODOS = {"Euler": paso_euler, "Punto medio": paso_punto_medio,
           "Heun": paso_heun, "RK4": paso_rk4}


def resolver(f, a, b, y0, n, paso):
    """Malla uniforme de n pasos; y0 puede ser escalar o vector real."""
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n debe ser un entero positivo")
    if not np.isfinite([a, b]).all() or b <= a:
        raise ValueError("Se requiere a < b, con extremos finitos")
    inicial = np.asarray(y0, dtype=float)
    if not np.isfinite(inicial).all():
        raise ValueError("La condición inicial debe ser finita")
    t = np.linspace(a, b, n + 1)
    h = (b - a) / n
    y = np.empty((n + 1,) + inicial.shape)
    y[0] = inicial
    for k in range(n):
        y[k + 1] = paso(f, t[k], y[k], h)
    return t, y




## Un paso calculado a mano


In [2]:
def f(t, y):
    return y - t**2 + 1


def exacta(t):
    return (t + 1)**2 - .5 * np.exp(t)


print("PVI: y' = y - t^2 + 1, y(0) = 0.5; h = 0.2")
for nombre, paso in METODOS.items():
    print(f"{nombre:12s}: y1 = {paso(f, 0., .5, .2):.10f}")
print(f"{'Exacta':12s}: y(0.2) = {exacta(.2):.10f}")



PVI: y' = y - t^2 + 1, y(0) = 0.5; h = 0.2
Euler       : y1 = 0.8000000000
Punto medio : y1 = 0.8280000000
Heun        : y1 = 0.8260000000
RK4         : y1 = 0.8292933333
Exacta      : y(0.2) = 0.8292986209


## Consistencia de Euler: residuo y defecto de un paso


El defecto se calcula partiendo de la solución exacta:
$$d_{k+1}=y(t_{k+1})-y(t_k)-h f(t_k,y(t_k)),\qquad \tau_{k+1}=d_{k+1}/h.$$
Consistencia significa $\max_k|\tau_{k+1}|\to0$ al refinar la malla.
Si $y\in C^2[a,b]$, Taylor da $|\tau_{k+1}|\leq Mh/2$, con $M=\max|y''|$.
Por eso Euler es consistente de orden uno y su defecto es de orden dos.
El siguiente experimento usa solo un paso en $t=0$; la prueba uniforme está en las notas.


In [3]:
# Para y'=y, y(0)=1, la solución exacta es exp(t).
# Se parte de un dato exacto en t=0; no se calcula el error global.
pasos_locales = np.array([.2, .1, .05, .025])
defectos_locales = np.expm1(pasos_locales) - pasos_locales
residuos_locales = defectos_locales / pasos_locales
print("\nConsistencia de Euler en t=0 para y'=y, y(0)=1")
print("   h          tau=d/h          d")
for h_local, tau_local, d_local in zip(pasos_locales, residuos_locales, defectos_locales):
    print(f"{h_local:7.3f} {tau_local:16.8e} {d_local:16.8e}")
p_tau = float(np.log2(residuos_locales[-2] / residuos_locales[-1]))
p_defecto = float(np.log2(defectos_locales[-2] / defectos_locales[-1]))
print(f"Orden observado de tau: {p_tau:.6f}; del defecto: {p_defecto:.6f}")
print("Experimento en un punto; la prueba de las notas usa una cota uniforme en toda la malla.")
assert abs(p_tau - 1) < .05 and abs(p_defecto - 2) < .05
assert np.all(residuos_locales <= .5 * np.exp(pasos_locales) * pasos_locales)




Consistencia de Euler en t=0 para y'=y, y(0)=1
   h          tau=d/h          d
  0.200   1.07013791e-01   2.14027582e-02
  0.100   5.17091808e-02   5.17091808e-03
  0.050   2.54219275e-02   1.27109638e-03
  0.025   1.26048210e-02   3.15120524e-04
Orden observado de tau: 1.012098; del defecto: 2.012098
Experimento en un punto; la prueba de las notas usa una cota uniforme en toda la malla.


## Refinamiento y orden observado


In [4]:
filas = []
for n in [10, 20, 40, 80]:
    fila = {"n": n, "h": 2 / n}
    for nombre, paso in METODOS.items():
        t, y = resolver(f, 0., 2., .5, n, paso)
        fila[nombre] = float(np.max(np.abs(y - exacta(t))))
    filas.append(fila)
print("\nErrores máximos sobre los nodos de [0,2]")
print(" n        Euler       Punto medio       Heun           RK4")
for fila in filas:
    print(f"{fila['n']:2d} " + " ".join(f"{fila[m]:14.6e}" for m in METODOS))
print("\nOrden observado log2(E_h / E_(h/2)), últimas dos mallas:")
ordenes = {m: float(np.log2(filas[-2][m] / filas[-1][m])) for m in METODOS}
for m, p in ordenes.items():
    print(f"{m:12s}: {p:.6f}")




Errores máximos sobre los nodos de [0,2]
 n        Euler       Punto medio       Heun           RK4
10   4.396874e-01   1.510249e-02   7.241732e-02   1.089498e-04
20   2.419719e-01   3.747074e-03   1.890478e-02   6.990307e-06
40   1.274657e-01   9.277142e-04   4.819865e-03   4.421339e-07
80   6.549505e-02   2.304037e-04   1.216136e-03   2.778991e-08

Orden observado log2(E_h / E_(h/2)), últimas dos mallas:
Euler       : 0.960652
Punto medio : 2.009517
Heun        : 1.986688
RK4         : 3.991850


## Modelo económico hipotético: acumulación de capital


In [5]:
delta, K0, T = .05, 10., 10.


def inversion(t):
    return 2 * np.exp(.02 * t)


def capital_rhs(t, K):
    return inversion(t) - delta * K


def capital_exacto(t):
    return K0 * np.exp(-delta * t) + 2 / (.05 + .02) * (
        np.exp(.02 * t) - np.exp(-delta * t))


print(f"\nCapital: K(10) exacto = {capital_exacto(T):.10f}")
for m, paso in METODOS.items():
    t, K = resolver(capital_rhs, 0., T, K0, 20, paso)
    print(f"{m:12s}: K(10) = {K[-1]:.10f}; error = {abs(K[-1]-capital_exacto(T)):.3e}")




Capital: K(10) exacto = 23.6330808385
Euler       : K(10) = 23.6791486574; error = 4.607e-02
Punto medio : K(10) = 23.6326201571; error = 4.607e-04
Heun        : K(10) = 23.6328425105; error = 2.383e-04
RK4         : K(10) = 23.6330808293; error = 9.137e-09


## Sistemas y estabilidad (complemento)


In [6]:
def oscilador(t, z):
    return np.array([z[1], -z[0]])


t, z = resolver(oscilador, 0., 1., [1., 0.], 40, paso_rk4)
referencia = np.column_stack([np.cos(t), -np.sin(t)])
print(f"\nSistema: error máximo RK4 = {np.max(np.abs(z-referencia)):.3e}")
for h in [.25, .75]:
    print(f"y'=-4y, h={h}: amplificación Euler={1-4*h:.2f}, "
          f"Heun={1-4*h+(-4*h)**2/2:.2f}")




Sistema: error máximo RK4 = 2.702e-09
y'=-4y, h=0.25: amplificación Euler=0.00, Heun=0.50
y'=-4y, h=0.75: amplificación Euler=-2.00, Heun=2.50


## Verificaciones y archivos para las transparencias


In [7]:
assert np.isclose(paso_euler(f, 0., .5, .2), .8)
assert np.isclose(paso_heun(f, 0., .5, .2), .826)
assert np.isclose(paso_punto_medio(f, 0., .5, .2), .828)
assert not np.isclose(paso_heun(f, 0., .5, .2), paso_punto_medio(f, 0., .5, .2))
for m, esperado in {"Euler": 1, "Punto medio": 2, "Heun": 2, "RK4": 4}.items():
    assert abs(ordenes[m] - esperado) < .12, (m, ordenes[m])
    assert all(filas[j + 1][m] < filas[j][m] for j in range(3))
    # Consistencia: integra una derivada constante exactamente.
    tt, yy = resolver(lambda t, y: np.ones_like(y), 0., 1., [2., 3.], 8, METODOS[m])
    np.testing.assert_allclose(yy[-1], [3., 4.], atol=1e-14)
assert np.max(np.abs(z - referencia)) < 4e-9

SALIDA = Path(__file__).resolve().parent / "resultados_edo_2026-2" if "__file__" in globals() else Path("resultados_edo_2026-2")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "verificacion.json").write_text(json.dumps({"errores": filas, "ordenes": ordenes}, indent=2))


def cientifico_tex(x):
    mantisa, exponente = f"{x:.3e}".split("e")
    return rf"${mantisa}\times10^{{{int(exponente)}}}$"


tabla = [r"\begin{tabular}{rrrrr}\toprule",
         r"$n$ & Euler & Punto medio & Heun & RK4\\\midrule"]
for fila in filas:
    tabla.append(str(fila["n"]) + " & " + " & ".join(cientifico_tex(fila[m]) for m in METODOS) + r"\\")
tabla.append(r"\bottomrule\end{tabular}")
(SALIDA / "tabla_errores.tex").write_text("\n".join(tabla))
tabla = [r"\begin{tabular}{lrr}\toprule", r"Método & Orden teórico & Orden observado\\\midrule"]
for (m, p), teorico in zip(ordenes.items(), [1, 2, 2, 4]):
    tabla.append(f"{m} & {teorico} & {p:.3f}" + r"\\")
tabla.append(r"\bottomrule\end{tabular}")
(SALIDA / "tabla_ordenes.tex").write_text("\n".join(tabla))
t, e = resolver(f, 0., 2., .5, 10, paso_euler)
_, h = resolver(f, 0., 2., .5, 10, paso_heun)
_, r = resolver(f, 0., 2., .5, 10, paso_rk4)
np.savetxt(SALIDA / "trayectorias.dat", np.column_stack([t, exacta(t), e, h, r]),
           header="t exacta Euler Heun RK4", comments="")
print("\nVerificación completa: pasos manuales, órdenes, consistencia y sistema vectorial.")



Verificación completa: pasos manuales, órdenes, consistencia y sistema vectorial.


## Actividad de clase: 80 evaluaciones de la función

1. Determine el número de pasos de cada método si el presupuesto es 80 evaluaciones de $f$.
2. Use `resolver` y la referencia exacta para comparar el error máximo sobre la malla.
3. Compruebe si el orden de precisión entre punto medio y Heun cambia en el modelo de capital.
4. Explique por qué el orden teórico no fija por sí solo el error para una malla dada.
5. Distinga el residuo normalizado $\tau=d/h$, el defecto de un paso $d$ y el error global.

## Trabajo posterior

- Repita la comparación para $y'=y$, $y(0)=1$, en $[0,2]$.
- Demuestre que punto medio y Heun dan la misma recurrencia en ese caso.
- Compare las aproximaciones finales obtenidas con $h$ y $h/2$.
  Bajo una expansión del error con término principal $Ch^p$, el error de la solución fina
  se estima por $|y_{h/2}(T)-y_h(T)|/(2^p-1)$. Es una estimación asintótica, no una cota certificada.
- Modifique el paso para $y'=-4y$ y observe cuándo se amplifican las perturbaciones.

## Referencias del curso

Capítulo 8: problema de Cauchy, métodos de un paso, Euler, trapecio y Runge–Kutta.
Capítulo 7: cuadratura. Clase 7C: modelo de acumulación de capital.
